# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset schema and metadata
dataset = mlc.Dataset(croissant_url)

# Display the Dataset's high-level metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Identifier: {getattr(dataset.metadata, 'identifier', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, their field IDs, and columns as per the Croissant schema.

Here we examine the metadata for all available record sets, and for each record set, we list the associated fields (columns) and their `@id`s.

In [ ]:
# List all record sets with their @id and field information
record_sets = list(dataset.iter_record_sets())
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set: {rs.id}")
    fields = rs.fields
    for field in fields:
        info = f"    Field: {field.id} (name='{field.name}', type='{field.data_type}')"
        print(info)
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to the record set and field `@id`s from the overview above.

For demonstration, we'll load the first available record set.

In [ ]:
# Collect all available record set IDs
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

# Load data for each record set
for record_set_id in record_set_ids:
    # The records generator returns dicts with field @id keys
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Demo with the first record set
main_record_set_id = record_set_ids[0]
print(f"Loaded DataFrame columns for {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing fields, and grouping records. All fields should be referenced using their `@id`s.

We will:
- Select a numeric field (e.g., molecular weight or normalized abundance, depending on schema, referenced by `@id`).
- Filter for entries where the value exceeds a threshold.
- Normalize the filtered field (z-score).
- Group by a categorical field (e.g., protein accession) if present.

In [ ]:
# Choose numeric and group field by their @id as identified above
df = dataframes[main_record_set_id]

# You may use the printout above to adjust these field names, using their exact @id
# For demonstration, we guess common field ids for proteomics datasets:
candidate_numeric_fields = [c for c in df.columns if "molecular_weight" in c.lower() or "normalized_abundance" in c.lower() or "mw" in c.lower() or "coverage" in c.lower() or "peptide_counts" in c.lower() or "intensity" in c.lower()]
if len(candidate_numeric_fields) > 0:
    numeric_field_id = candidate_numeric_fields[0]
else:
    numeric_field_id = df.columns[df.dtypes.apply(lambda x: pd.api.types.is_numeric_dtype(x))][0]

print(f"Numeric field selected for EDA: {numeric_field_id}")

group_field_candidates = [c for c in df.columns if 'accession' in c.lower() or 'protein' in c.lower() or 'sample' in c.lower()]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
else:
    group_field_id = None

# Drop NA for numeric field
df_eda = df.copy()
df_eda = df_eda.dropna(subset=[numeric_field_id])

# Convert to float if possible
df_eda[numeric_field_id] = pd.to_numeric(df_eda[numeric_field_id], errors='coerce')
df_eda = df_eda.dropna(subset=[numeric_field_id])

threshold = df_eda[numeric_field_id].mean()  # example threshold == mean
filtered_df = df_eda[df_eda[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a field if found
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use Matplotlib and Seaborn to plot histograms and boxplots of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
sns.histplot(df_eda[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id and group_field_id in df_eda.columns:
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=df_eda, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² Croissant dataset, listed its available record sets, and processed proteomics data by referencing fields and records using their `@id`s. We demonstrated basic filtering, normalization, and grouping. Visualization provides insight into underlying data distributions, which is essential for downstream bioinformatics or ML workflows.

Explore the schema further and adapt field IDs for your analytic goals. For full details, see the [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).